In [1]:
import torch

In [3]:
if torch.cuda.is_available():
    device = 'gpu'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

device

'cpu'

In [4]:
from pathlib import Path

batch_chorales_dir = Path("jsb_chorales")

In [10]:
print(Path(batch_chorales_dir / "train" / "chorale_000.csv").read_text()[:120] + "...")

note0,note1,note2,note3
74,70,65,58
74,70,65,58
74,70,65,58
74,70,65,58
75,70,58,55
75,70,58,55
75,70,60,55
75,70,60,55
...


In [11]:
import csv

def load_chorales(batch_chorales_dir):
    chorales = {"train":[], "test":[], "valid":[]}
    for subset, chorale_list in chorales.items():
        for chorale_path in sorted((batch_chorales_dir / subset).glob("*.csv")):
            with chorale_path.open() as f:
                reader = csv.reader(f)
                next(reader)
                chorale = [[int(note) for note in row] for row in reader]
                chorale_list.append(torch.tensor(chorale))
    return chorales

In [12]:
chorales = load_chorales(batch_chorales_dir)

In [15]:
from IPython.display import Audio

def notes_to_frequencies(notes):
    # Frequency doubles when you go up one octave; there are 12 semi-tones
    # per octave; Note A on octave 4 is 440 Hz, and it is note number 69.
    return 2 ** ((notes - 69) / 12) * 440

def frequencies_to_samples(frequencies, tempo, sample_rate):
    note_duration = 60 / tempo # the tempo is measured in beats per minutes
    # To reduce click sound at every beat, we round the frequencies to try to
    # get the amplitude close to zero at the end of each note.
    frequencies = (note_duration * frequencies).round() / note_duration
    n_samples = int(note_duration * sample_rate)
    time = torch.linspace(0, note_duration, n_samples)
    sine_waves = torch.sin(2 * torch.pi * frequencies.reshape(-1, 1) * time)
    # Removing all notes with frequencies ≤ 9 Hz (includes note 0 = silence)
    sine_waves *= (frequencies > 9.).reshape(-1, 1)
    return sine_waves.reshape(-1)

def chords_to_samples(chords, tempo, sample_rate):
    freqs = notes_to_frequencies(chords)
    freqs = torch.cat([freqs, freqs[-1:]], dim=0) # make last note a bit longer
    merged = torch.stack([frequencies_to_samples(melody, tempo, sample_rate)
                          for melody in freqs.T]).mean(dim=0)
    n_fade_out_samples = sample_rate * 60 // tempo # fade out last note
    fade_out = torch.linspace(1., 0., n_fade_out_samples)**2
    merged[-n_fade_out_samples:] *= fade_out
    return merged

def play_chords(chords, tempo=160, amplitude=0.1, sample_rate=44100):
    samples = amplitude * chords_to_samples(chords, tempo, sample_rate)
    return display(Audio(samples, rate=sample_rate))

In [16]:
for index in range(3):
    play_chords(chorales["train"][index])

In [139]:
x = chorales['train'][0]
print(x.shape)
print(x[:12])
print(x.dtype)

torch.Size([192, 4])
tensor([[74, 70, 65, 58],
        [74, 70, 65, 58],
        [74, 70, 65, 58],
        [74, 70, 65, 58],
        [75, 70, 58, 55],
        [75, 70, 58, 55],
        [75, 70, 60, 55],
        [75, 70, 60, 55],
        [77, 69, 62, 50],
        [77, 69, 62, 50],
        [77, 69, 62, 50],
        [77, 69, 62, 50]])
torch.int64


In [127]:
# Build Vocabulary
unique_tokens = set()

for ch in chorales['train']:
    unique_tokens.update(
        ch.flatten().tolist()
    )

vocab = sorted(unique_tokens)
print(vocab)
print(len(vocab))

[0, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81]
47


In [129]:
stoi = {token : idx for idx, token in enumerate(vocab)}
print(stoi)

{0: 0, 36: 1, 37: 2, 38: 3, 39: 4, 40: 5, 41: 6, 42: 7, 43: 8, 44: 9, 45: 10, 46: 11, 47: 12, 48: 13, 49: 14, 50: 15, 51: 16, 52: 17, 53: 18, 54: 19, 55: 20, 56: 21, 57: 22, 58: 23, 59: 24, 60: 25, 61: 26, 62: 27, 63: 28, 64: 29, 65: 30, 66: 31, 67: 32, 68: 33, 69: 34, 70: 35, 71: 36, 72: 37, 73: 38, 74: 39, 75: 40, 76: 41, 77: 42, 78: 43, 79: 44, 80: 45, 81: 46}


In [131]:
itos = {idx : token for token, idx in stoi.items()}
print(itos)

{0: 0, 1: 36, 2: 37, 3: 38, 4: 39, 5: 40, 6: 41, 7: 42, 8: 43, 9: 44, 10: 45, 11: 46, 12: 47, 13: 48, 14: 49, 15: 50, 16: 51, 17: 52, 18: 53, 19: 54, 20: 55, 21: 56, 22: 57, 23: 58, 24: 59, 25: 60, 26: 61, 27: 62, 28: 63, 29: 64, 30: 65, 31: 66, 32: 67, 33: 68, 34: 69, 35: 70, 36: 71, 37: 72, 38: 73, 39: 74, 40: 75, 41: 76, 42: 77, 43: 78, 44: 79, 45: 80, 46: 81}


In [132]:
encoded_chorales = {
    "train" : [],
    "test" : [],
    "valid" : []
}

for split in encoded_chorales:
    for ch in chorales[split]:
        encoded = torch.tensor([
            [stoi[int(token)] for token in timestep] for timestep in ch
        ], dtype=torch.long)
        encoded_chorales[split].append(encoded)

In [137]:
chorales['train'][0][:12]

tensor([[74, 70, 65, 58],
        [74, 70, 65, 58],
        [74, 70, 65, 58],
        [74, 70, 65, 58],
        [75, 70, 58, 55],
        [75, 70, 58, 55],
        [75, 70, 60, 55],
        [75, 70, 60, 55],
        [77, 69, 62, 50],
        [77, 69, 62, 50],
        [77, 69, 62, 50],
        [77, 69, 62, 50]])

In [140]:
encoded_chorales['train'][0][:12]

tensor([[39, 35, 30, 23],
        [39, 35, 30, 23],
        [39, 35, 30, 23],
        [39, 35, 30, 23],
        [40, 35, 23, 20],
        [40, 35, 23, 20],
        [40, 35, 25, 20],
        [40, 35, 25, 20],
        [42, 34, 27, 15],
        [42, 34, 27, 15],
        [42, 34, 27, 15],
        [42, 34, 27, 15]])

In [164]:
class CharaleDataset(torch.utils.data.Dataset):
    def __init__(self, chorales, window_len = 32):
        self.samples = []
        for ch in chorales:
            ch = ch.view(-1)
            for i in range(len(ch) - window_len):
                            x = ch[i: i + window_len]
                            y = ch[i + 1 : i + window_len + 1]


                            self.samples.append((x, y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]
        


In [205]:
c = torch.arange(1, 20)
window_len = 17
for i in range(len(c) - window_len):
    x = c[i: i + window_len]
    y = c[i + 1 : i + window_len + 1]
    print(i)
    print('x', x)
    print('y', y)
    


0
x tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])
y tensor([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18])
1
x tensor([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18])
y tensor([ 3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19])


In [193]:
x

tensor([[72, 64, 55, 48],
        [72, 64, 55, 48]])

In [195]:
y

tensor([[72, 64, 55, 48],
        [72, 64, 57, 48]])

In [194]:
len(c)

19

In [182]:
window_len = 32
train_dataset = CharaleDataset(
    encoded_chorales['train'],
    window_len
)
valid_dataset = CharaleDataset(
    encoded_chorales['valid'],
    window_len
)
test_dataset = CharaleDataset(
    encoded_chorales['test'],
    window_len
)
    

In [187]:
x,y = train_dataset[0]

In [188]:
x

tensor([39, 35, 30, 23, 39, 35, 30, 23, 39, 35, 30, 23, 39, 35, 30, 23, 40, 35,
        23, 20, 40, 35, 23, 20, 40, 35, 25, 20, 40, 35, 25, 20])

In [189]:
y

tensor([35, 30, 23, 39, 35, 30, 23, 39, 35, 30, 23, 39, 35, 30, 23, 40, 35, 23,
        20, 40, 35, 23, 20, 40, 35, 25, 20, 40, 35, 25, 20, 42])

In [ ]:
class CharaleDataset(torch.utils.data.Dataset):
    def __init__(self, chorales, window_len = 32):
        self.samples = []
        for ch in chorales:
            for i in range(len(ch) - window_len):
                x = ch[i: i + window_len]
                y = ch[i + 1 : i + window_len + 1]

                self.samples.append((x, y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]
        
